In [0]:
%python
from pyspark.sql import SparkSession
# need a databrick jobs to repeate trigger and process all unprocessed files

spark = SparkSession.builder.appName("AutoLoaderProductionPipeline").getOrCreate()


# configure for access the volume
source_volume_path = "/Volumes/dbs_study/landing/customers_vol/customers_info/"

# recommnednt to use a relative close location to volume
# use the checkpoint that rock db can write the file process data from spark scheuler to memory, and the processed file
checkpoint_base = "/Volumes/dbs_study/landing/customers_vol/_internal_metadata/customers"
schema_location = f"{checkpoint_base}/schema"
checkpoint_location = f"{checkpoint_base}/checkpoint"

# Target production table
target_table = "dbs_study.landing.customers_v2"


#READ STREAM WITH AUTO LOADER (cloudFiles)

autoloader_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    
    #  Schema Management
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Options: addNewColumns, failOnNewColumns, rescue
    
    # Performance Tuning & Backfill Safety
    .option("cloudFiles.maxFilesPerTrigger", 1000)     # Limits files(maximum 1000) processed per micro-batch
    .option("cloudFiles.maxFileAge", "14 days")        # Keeps the RocksDB state store slim, in reality, kept longer if need for audit if the budge is allowed
    .option("cloudFiles.maxBytesPerTrigger", "10gb") # Limits bytes processed per micro-batch: here maximum process is 10 gb
    # File Parsing Options
    .option("header", "true")
    .option("inferSchema", "true")                     # Automatically infers data types of incoming values
    # File Path Options
    .option("cloudFiles.useNotifications", "true")
    .load(source_volume_path)
)


# WRITE STREAM TO TARGET DELTA TABLE
query = (
    autoloader_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    # allow to merge the schema for the schema evolution
    .option("mergeSchema", "true")
    .outputMode("append")
    
    
    # --- Trigger Options ---
    # Trigger.AvailableNow processes all available files as a batch and shuts down the cluster: only proces the file for once
    .trigger(availableNow=True) 
    
    # Alternatively, use continuous or processing time triggers:
    # .trigger(processingTime='1 minute') 
    # if you don't set an interval at all, Structured Streaming defaults to checking for new data every few milliseconds, which can generate a high volume of cloud storage API calls per day and unexpected charges.

    # by default_ta
    .toTable(target_table)
)

# Wait for the streaming query to finish processing before exiting the cell
query.awaitTermination()

In [0]:
%sql

use catalog dbs_study;

select * from dbs_study.landing.customers_v2 limit 100;